# Ferienakademie 2026 — feature showcase

This second demo is meant for **discussion of the model**, not as a good solution.
It deliberately combines more of the available mechanisms at once:

- internal walls and local vision
- three agent types
- persistent per-agent memory
- movement and pushing in the same turn
- finite batteries with continuous recharge
- three pheromone channels with different decay times and colors

The controller below is intentionally crude. The agents explore, react to local cargo/target sightings, leave different signals, and attempt to transport the cargo. The run does not need to solve the scenario.

In [ ]:
from fa2026 import Action, Cell, Config, Pheromone, execute, load_scenario

scenario = load_scenario("showcase")
scenario.rules

## Controller

The pheromone meanings below are chosen only for this demo:

- **blue**: scout/exploration trail
- **orange**: cargo seen / recruitment
- **magenta**: target seen

Agent types use the same `act()` function but different parameters and priorities. Type 0 agents mainly explore, type 1 agents are stronger cargo carriers, and type 2 agents react more strongly to target signals.

In [ ]:
import math
import numpy as np


def setup(rules):
    return Config(
        pheromones=(
            Pheromone(decay=0.05, color="deepskyblue"),
            Pheromone(decay=0.10, color="orange"),
            Pheromone(decay=0.18, color="magenta"),
        ),
        # 35 type-1 carriers, 25 type-2 relays; the remaining 60 are type 0.
        agent_type_counts=(35, 25),
        initial_memory=(
            (0.0, 9.0, 0.0, 0.0, 0.0, 1.0),
            (2.1, 9.0, 0.0, 0.0, 0.0, 1.0),
            (4.2, 9.0, 0.0, 0.0, 0.0, 1.0),
        ),
        # move step, push strength, trail emission
        parameters=(
            (0.65, 0.45, 0.035),  # scouts
            (0.48, 1.10, 0.020),  # carriers
            (0.56, 0.70, 0.025),  # relays
        ),
    )


def visible_vector(observation, flag):
    mask = (observation.vision & int(flag)) != 0
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None
    center = observation.vision.shape[0] // 2
    dx = xs - center + 0.5 - observation.cell_position[0]
    dy = ys - center + 0.5 - observation.cell_position[1]
    return np.array([float(np.mean(dx)), float(np.mean(dy))])


def unit(vector):
    length = float(np.linalg.norm(vector))
    return vector / length if length > 1e-12 else np.zeros(2)


def pheromone_direction(field):
    if field.size == 0 or float(np.max(field)) <= 1e-8:
        return None
    py, px = np.unravel_index(np.argmax(field), field.shape)
    direction = np.array([px - 1, py - 1], dtype=float)
    return unit(direction) if np.linalg.norm(direction) > 0 else None


def avoid_wall(observation, move):
    """Very crude one-cell wall avoidance, deliberately not path planning."""
    move = np.asarray(move, dtype=float).copy()
    center = observation.vision.shape[0] // 2
    sx = 1 if move[0] > 0 else (-1 if move[0] < 0 else 0)
    sy = 1 if move[1] > 0 else (-1 if move[1] < 0 else 0)

    if sx and int(observation.vision[center, center + sx]) & int(Cell.WALL):
        move[0] = 0.0
    if sy and int(observation.vision[center + sy, center]) & int(Cell.WALL):
        move[1] = 0.0
    if np.linalg.norm(move) < 1e-12:
        move = np.array([-sy, sx], dtype=float)
    return move


def act(observation, memory, agent_type, config):
    memory = memory.copy()
    step, push_strength, trail = config.parameters[agent_type]

    cargo = visible_vector(observation, Cell.CARGO)
    target = visible_vector(observation, Cell.TARGET)

    # Memory: exploration angle, countdown, remembered target direction (x/y),
    # remaining target-memory time, initialized flag.
    angle = float(memory[0])
    countdown = float(memory[1]) - 1.0
    if countdown <= 0:
        angle = (angle + 2.399963229728653) % (2 * math.pi)
        countdown = 9.0 + 2.0 * agent_type
    explore = np.array([math.cos(angle), math.sin(angle)])

    if target is not None:
        remembered = unit(target)
        memory[2:4] = remembered
        memory[4] = 24.0
    elif memory[4] > 0:
        memory[4] -= 1.0
        remembered = unit(memory[2:4])
    else:
        remembered = None

    emissions = np.zeros(3, dtype=float)

    # Scouts leave a faint trail almost continuously.
    if agent_type == 0:
        emissions[0] = trail

    # Seeing the target produces a short-lived target signal.
    if target is not None:
        emissions[2] = 0.08

    # Once inside the cargo, move and push together. If no target direction is
    # known, push in the current exploration direction.
    on_cargo = cargo is not None and abs(cargo[0]) <= 1.05 and abs(cargo[1]) <= 1.05
    if on_cargo:
        if target is not None:
            direction = unit(target - cargo)
        elif remembered is not None:
            direction = remembered
        else:
            direction = explore
        emissions[1] = 0.10
        memory[0] = math.atan2(direction[1], direction[0])
        memory[1] = countdown
        return Action(
            move=tuple(0.7 * step * direction),
            push=tuple(push_strength * direction),
            pheromones=tuple(emissions),
        ), memory

    # Cargo sighting recruits nearby agents through the orange channel.
    if cargo is not None:
        emissions[1] = 0.08
        move = step * unit(cargo)
    else:
        move = None

        # Carriers preferentially follow cargo/recruitment signal.
        if agent_type == 1:
            direction = pheromone_direction(observation.pheromones[1])
            if direction is not None:
                move = step * direction

        # Relays preferentially follow target signal.
        if move is None and agent_type == 2:
            direction = pheromone_direction(observation.pheromones[2])
            if direction is not None:
                move = step * direction

        # Everyone can weakly follow scout trails; otherwise explore.
        if move is None:
            direction = pheromone_direction(observation.pheromones[0])
            move = step * (direction if direction is not None else explore)

    move = avoid_wall(observation, move)
    if np.linalg.norm(move) > 0:
        move = step * unit(move)
        memory[0] = math.atan2(move[1], move[0])
    memory[1] = countdown

    return Action(move=tuple(move), pheromones=tuple(emissions)), memory


## Watch the feature-rich run

Agent types are colored automatically in the visualization. Pheromone colors come from the student's `Pheromone(...)` definitions. Where channels overlap, their display colors are blended.

This run intentionally stops after a fixed number of turns if it has not solved the task.

In [ ]:
result = execute(
    scenario,
    setup,
    act,
    visualize=True,
    delay=0.0,
    draw_every=5,
    max_turns=350,
)

if result.success:
    print(f"Success after {result.turns} turns.")
else:
    print(f"Stopped after {result.turns} turns without success.")

The point of this notebook is not the quality of the controller. It is a compact way to inspect how walls, local sensing, energy, several pheromone channels, agent types, memory, movement and cargo manipulation interact under the current rules.